In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import anthropic
import pandas as pd
from datasets import load_from_disk
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

/Users/faizanfaisal/Laptop_Data/masters/masters_codebases/LEGAL-UQA/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
dataset_dict = load_from_disk("../../data/complete_dataset/")

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['question_eng', 'question_urdu', 'context_eng', 'context_urdu', 'answer_eng', 'answer_urdu', 'context_index', '__index_level_0__'],
        num_rows: 495
    })
    validation: Dataset({
        features: ['question_eng', 'question_urdu', 'context_eng', 'context_urdu', 'answer_eng', 'answer_urdu', 'context_index', '__index_level_0__'],
        num_rows: 124
    })
})


In [3]:
validation_df = pd.DataFrame(dataset_dict["validation"])
df = validation_df[["question_urdu", "answer_urdu"]].copy()
df["id"] = range(len(df))
df.rename(
    columns={"question_urdu": "question", "answer_urdu": "answers"},
    inplace=True,
)
df = df[["id", "question", "answers"]]

display(df)

,id,question,answers
0,0,اسلامی کونسل کے رکن کی مدت کار کتنی ہوتی ہے؟,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...
1,1,پاکستان میں یوم آغاز سے قبل کن علاقوں کو قبائل...,پاکستان میں یوم آغاز سے قبل قبائلی علاقے جات م...
2,2,کیا کسی عدالت عالیہ کے جج کا اس کی رضامندی کے ...,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...
3,3,موجودہ قواعد و احکام میں ترمیم کرنے کا اختیار ...,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...
4,4,پارٹی کے سربراہ کو کسی رکن کو منحرف قرار دینے ...,کسی رکن کو منحرف قرار دینے سے پہلے، پارٹی کے س...
...,...,...,...
119,119,مشترکہ مفادات کی کونسل کا چیئرمین کون ہوتا ہے؟,وزیر اعظم مشترکہ مفادات کی کونسل کا چیئرمین ہو...
120,120,کیا کسی شخص کو شہادت حاصل کرنے کے لئے اذیت دین...,نہیں، کسی شخص کو شہادت حاصل کرنے کی غرض سے اذی...
121,121,سینیٹ میں اسپیکر اور ڈپٹی اسپیکر کے مساوی عہدے...,سینیٹ میں اسپیکر اور ڈپٹی اسپیکر کے مساوی عہدے...
122,122,مالی بل اپنی قانون سازی کا عمل کہاں سے شروع کر...,مالی بل کی ابتدا قومی اسمبلی میں ہوگی۔


In [4]:
MODEL_NAME = "claude-sonnet-5"
MODEL_VERSION = "claude-sonnet-5"  # Canonical pinned model ID (Anthropic 4.6+)
MAX_WORKERS = 16
client = anthropic.Anthropic(max_retries=5, timeout=120.0)


def get_urdu_response(context, question):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        thinking={"type": "disabled"},
        system="You are a Pakistani legal expert who speaks in Urdu. You only respond in Urdu script. Do not include any internal reasoning or XML tags in your response.",
        messages=[
            {"role": "user", "content": f"""Below is a context provided, followed by a question. Your task is to answer the question based on the given context. The answer should consist only of the response to the question in Urdu and should not include any other information or commentary.

    Context: ```{context}```

    Question: ```{question}```

    Answer:"""}
        ],
    )

    return message.content[0].text

In [5]:
validation_examples = list(dataset_dict["validation"])
predicted_answers = [None] * len(validation_examples)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_index = {
        executor.submit(
            get_urdu_response,
            example["context_urdu"],
            example["question_urdu"],
        ): index
        for index, example in enumerate(validation_examples)
    }

    for future in tqdm(
        as_completed(future_to_index),
        total=len(future_to_index),
        desc=f"Processing {MODEL_NAME} responses",
        unit="item",
    ):
        index = future_to_index[future]
        predicted_answers[index] = future.result()

assert all(answer is not None for answer in predicted_answers)


Processing claude-sonnet-5 responses:   0%|          | 0/124 [00:00<?, ?item/s]


Processing claude-sonnet-5 responses:   1%|          | 1/124 [00:01<03:18,  1.61s/item]


Processing claude-sonnet-5 responses:   2%|▏         | 3/124 [00:01<00:57,  2.09item/s]


Processing claude-sonnet-5 responses:   7%|▋         | 9/124 [00:02<00:17,  6.70item/s]


Processing claude-sonnet-5 responses:   9%|▉         | 11/124 [00:02<00:16,  7.04item/s]


Processing claude-sonnet-5 responses:  11%|█▏        | 14/124 [00:02<00:13,  8.38item/s]

... (55 outputs left out)


Processing claude-sonnet-5 responses:  98%|█████████▊| 121/124 [00:16<00:00, 10.64item/s]


Processing claude-sonnet-5 responses:  99%|█████████▉| 123/124 [00:17<00:00,  5.40item/s]


Processing claude-sonnet-5 responses: 100%|██████████| 124/124 [00:18<00:00,  2.70item/s]


Processing claude-sonnet-5 responses: 100%|██████████| 124/124 [00:18<00:00,  6.69item/s]

In [6]:
predicted_answers[:5]

['تین سال',
 'بلوچستان اور خیبر پختونخوا کے قبائلی علاقے جات، اور امب، چترال، دیر اور سوات کی سابقہ ریاستیں۔',
 'نہیں، کسی جج کا تبادلہ اس کی رضامندی کے بغیر نہیں کیا جائے گا۔',
 'وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت کے پاس موجودہ قواعد و احکام میں وقتاً فوقتاً ترمیم کرنے کا اختیار ہے۔',
 'پارٹی کے سربراہ کو اعلان کرنے سے پہلے مذکورہ رکن کو اس بارے میں اظہار وجوہ کا موقع فراہم کرنا چاہیے کہ کیوں نہ اس کے خلاف انحراف کا اعلان کر دیا جائے۔']

In [7]:
df["requested_model"] = MODEL_NAME
df["model_version"] = MODEL_VERSION
df["reasoning_mode"] = "disabled"
df["prediction_text_claude_sonnet_5"] = predicted_answers

In [8]:
display(df.head())

,id,question,answers,requested_model,model_version,reasoning_mode,prediction_text_claude_sonnet_5
0,0,اسلامی کونسل کے رکن کی مدت کار کتنی ہوتی ہے؟,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...,claude-sonnet-5,claude-sonnet-5,disabled,تین سال
1,1,پاکستان میں یوم آغاز سے قبل کن علاقوں کو قبائل...,پاکستان میں یوم آغاز سے قبل قبائلی علاقے جات م...,claude-sonnet-5,claude-sonnet-5,disabled,بلوچستان اور خیبر پختونخوا کے قبائلی علاقے جات...
2,2,کیا کسی عدالت عالیہ کے جج کا اس کی رضامندی کے ...,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...,claude-sonnet-5,claude-sonnet-5,disabled,نہیں، کسی جج کا تبادلہ اس کی رضامندی کے بغیر ن...
3,3,موجودہ قواعد و احکام میں ترمیم کرنے کا اختیار ...,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...,claude-sonnet-5,claude-sonnet-5,disabled,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...
4,4,پارٹی کے سربراہ کو کسی رکن کو منحرف قرار دینے ...,کسی رکن کو منحرف قرار دینے سے پہلے، پارٹی کے س...,claude-sonnet-5,claude-sonnet-5,disabled,پارٹی کے سربراہ کو اعلان کرنے سے پہلے مذکورہ ر...


In [9]:
df.to_csv("../../data/model_qa_responses/claude-sonnet-5-responses.csv", index=False)